In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as filter
import random


In [4]:
# transformer_blocks.py

import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------
# Self-Attention Head
# ----------------------------
class SelfAttentionHead(nn.Module):
    def __init__(self, embedding_dim, block_size, head_size):
        super().__init__()
        self.key = nn.Linear(embedding_dim, head_size, bias=False)
        self.query = nn.Linear(embedding_dim, head_size, bias=False)
        self.value = nn.Linear(embedding_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) / (C ** 0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)
        out = wei @ v
        return out

# ----------------------------
# Multi-Head Attention
# ----------------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, block_size, num_heads):
        super().__init__()
        head_size = embedding_dim // num_heads
        self.heads = nn.ModuleList([SelfAttentionHead(embedding_dim, block_size, head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, embedding_dim)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

# ----------------------------
# Feed Forward Network
# ----------------------------
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd)
        )
    def forward(self, x):
        return self.net(x)

# ----------------------------
# Transformer Block
# ----------------------------
class Block(nn.Module):
    def __init__(self, embedding_dim, block_size, n_heads):
        super().__init__()
        self.sa = MultiHeadAttention(embedding_dim, block_size, n_heads)
        self.ffwd = FeedForward(embedding_dim)
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [18]:
corpus = [
    "I started learning Python last month.",
    "Python makes coding really easy and fun.",
    "I use Python for small automation tasks.",
    "Python works great for web development.",
    "Many beginners choose Python as their first language.",
    "I wrote my first game using Python.",
    "Python has a lot of useful libraries.",
    "You can build AI projects with Python.",
    "Debugging Python code is usually simple.",
    "Python is one of my favorite programming languages."
]
#we have prepare data transformer model

# for indicating eng of sentence we have to use <END> and then we join sentences

corpus = [i + " <END>" for i in corpus]
text = " ".join(corpus)
# we have to converet dat into number
#for this we use tokeniazation
words = list(set(text.split()))
words2idx ={w:i for i,w in enumerate(words)}
print(words2idx)
'''
{'favorite': 0, 'usually': 1, 'development.': 2, 'started': 3,
'as': 4, 'Many': 5, 'useful': 6, 'lot': 7, '<END>': 8, 'projects': 9,
'learning': 10, 'has': 11, 'with': 12, 'AI': 13, 'code': 14,
 'is': 15, 'really': 16, 'wrote': 17, 'a': 18, 'of': 19, 'first': 20,
 'language.': 21, 'game': 22, 'Debugging': 23, 'coding': 24, 'easy': 25,
 'build': 26, 'one': 27, 'their': 28, 'Python.': 29, 'programming': 30,
 'languages.': 31, 'automation': 32, 'libraries.': 33, 'web': 34,
 'month.': 35, 'and': 36, 'Python': 37, 'I': 38, 'great': 39, 'choose': 40,
 'last': 41, 'beginners': 42, 'works': 43, 'my': 44, 'using': 45, 'fun.': 46,
  'can': 47, 'You': 48, 'tasks.': 49, 'simple.': 50, 'small': 51, 'makes': 52, 'for': 53, 'use': 54}

'''
#we are just take token for each number


{'favorite': 0, 'usually': 1, 'development.': 2, 'started': 3, 'as': 4, 'Many': 5, 'useful': 6, 'lot': 7, '<END>': 8, 'projects': 9, 'learning': 10, 'has': 11, 'with': 12, 'AI': 13, 'code': 14, 'is': 15, 'really': 16, 'wrote': 17, 'a': 18, 'of': 19, 'first': 20, 'language.': 21, 'game': 22, 'Debugging': 23, 'coding': 24, 'easy': 25, 'build': 26, 'one': 27, 'their': 28, 'Python.': 29, 'programming': 30, 'languages.': 31, 'automation': 32, 'libraries.': 33, 'web': 34, 'month.': 35, 'and': 36, 'Python': 37, 'I': 38, 'great': 39, 'choose': 40, 'last': 41, 'beginners': 42, 'works': 43, 'my': 44, 'using': 45, 'fun.': 46, 'can': 47, 'You': 48, 'tasks.': 49, 'simple.': 50, 'small': 51, 'makes': 52, 'for': 53, 'use': 54}
